Olivier Meyers 08-10-2025 - jupyter notebook for the training of a CNN U-Net for the denoising of microseismic bedretto data, code inspired and/or borrowed from Nikolaj Dahmen's MarsQuakeNet. Cell

In [ ]:
import os
# Root of the local Bedretto dataset (not distributed with this repo, see README)
DATA_DIR = os.environ.get("BEDRETTO_DATA_DIR", "data") + "/"

In [ ]:
import glob

from scipy.signal import stft, istft
import numpy as np

import matplotlib.pyplot as plt
from matplotlib import gridspec

import time
from datetime import datetime
from sklearn.preprocessing import RobustScaler, StandardScaler

import tensorflow as tf
import random
#from tensorflow.keras import Input, Model
#from tensorflow.keras.layers import Activation, BatchNormalization, Conv2D,Conv2DTranspose, MaxPool2D,Concatenate, AveragePooling2D
#from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping
from obspy.signal.rotate import rotate_ne_rt
import subprocess
import scipy
import re

In [ ]:
from tensorflow.keras import Input, Model
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Activation, BatchNormalization, Conv2D,Conv2DTranspose, MaxPool2D,Concatenate, AveragePooling2D
from tensorflow.keras.callbacks import ModelCheckpoint,EarlyStopping

In [ ]:
print("TensorFlow version:", tf.__version__)
print("CUDA available:", tf.test.is_built_with_cuda())
print("GPU devices:", tf.config.list_physical_devices('GPU'))

# Try to explicitly enable GPU memory growth
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print("Memory growth setting error:", e)

# Run a simple test
if tf.config.list_physical_devices('GPU'):
    with tf.device('/GPU:0'):
        x = tf.random.normal([1000, 1000])
        y = tf.random.normal([1000, 1000])
        import time
        start = time.time()
        z = tf.matmul(x, y)
        _ = z.numpy()  # Force execution
        print(f"GPU Matrix multiplication took: {time.time() - start:.4f} seconds")
else:
    print("No GPU available, using CPU instead.")
    import time
    start = time.time()
    x = tf.random.normal([1000, 1000])
    y = tf.random.normal([1000, 1000])
    z = tf.matmul(x, y)
    _ = z.numpy()
    print(f"CPU Matrix multiplication took: {time.time() - start:.4f} seconds")

In [ ]:
# Check TensorFlow GPU Availability
def check_gpu_availability():
    num_gpus = len(tf.config.list_physical_devices('GPU'))
    print("Num GPUs Available: ", num_gpus)

    if num_gpus == 0:
        print('Not connected to a GPU')
    else:
        print('GPU is available.')

# Check GPU details using nvidia-smi
def get_gpu_info():
    try:
        result = subprocess.run(['nvidia-smi'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        gpu_info = result.stdout
        if 'failed' in gpu_info.lower():
            print('Not connected to a GPU')
        else:
            print('GPU Info:\n', gpu_info)
    except FileNotFoundError:
        print("nvidia-smi command not found. Please ensure NVIDIA drivers are installed.")

# List local devices using TensorFlow
def list_local_devices():
    from tensorflow.python.client import device_lib
    devices = device_lib.list_local_devices()
    print("Local devices:")
    for device in devices:
        print(device)

# Execute the functions
check_gpu_availability()
get_gpu_info()
list_local_devices()
# plot settings
plt.rcParams['figure.figsize'] = [16, 12]
plt.rcParams['font.size'] = 12

In [ ]:
def normalize_percentile(data,quantile_range=(25,75),unit_variance=False,limit=50,clipping='none',scaler="standard"): #changed clipping to 'none' and scaler to "standard"
  """
  Robust normalisation of samples  1-component data
  For data with outliers (w hard/soft clipping), seperately for real and imag.
  limit = 50
  Input: data and optional arguments
  Output: normalised data
  Requires: sklearn Robustscaler
  """
  # get data
  data_real = data[:,:,0]  # real
  data_imag = data[:,:,1]  # imag

  freq_bins, time_bins = data_real.shape  
  # define scaler
  if scaler == "robust":
    scaler =  RobustScaler(unit_variance=unit_variance)
  else:
    scaler = StandardScaler()

  # apply to data: first flatten and then reshape
  data_real_norm = scaler.fit_transform(np.expand_dims(data_real.flatten(),1)).reshape((freq_bins, time_bins)) # shape of the stfts
  data_imag_norm = scaler.fit_transform(np.expand_dims(data_imag.flatten(),1)).reshape((freq_bins, time_bins))


  if clipping=='constant':  # hard clipping, all values > limit
    data_real_norm[data_real_norm>limit]=limit
    data_real_norm[data_real_norm<-limit]=-limit
    data_imag_norm[data_imag_norm>limit]=limit
    data_imag_norm[data_imag_norm<-limit]=-limit

  # combine real and imag
  data_return = np.zeros(data.shape)
  data_return[:,:,0] = data_real_norm
  data_return[:,:,1] = data_imag_norm

  return data_return

# def fix_timebins(x, target_T=192):
#     # x: (F, T) complex
#     F, T = x.shape
#     if T == target_T:
#         return x
#     if T > target_T:
#         trim = T - target_T
#         left = trim // 2
#         right = trim - left
#         return x[:, left:T-right]
#     # pad
#     pad = target_T - T
#     left = pad // 2
#     right = pad - left
#     return np.pad(x, ((0,0),(left,right)), mode='constant')


def map_func(event_file, noise_file, clip_val=1000, channel_norm=True, sps=100000, scaler="robust"):
    """
    Mapping (numpy) function in input_pipeline: load data and normalise
    """
    len_sample = 4967
    noise_tmp = np.load(noise_file, allow_pickle=True) #6500 samples long
    min_snr = 0.2 
    delta = 0
    ampl_fac = None # TEMP
    
    stft_parameters = {"nperseg": 64, "nfft": 126, "fs": 100000, "noverlap": 38, "boundary": "zeros", "padded": False}
    #num_freq_bins = int(stft_parameters["nfft"] / 2 + 1)
    #bin_above1Hz = np.argmax(np.linspace(0, 50000, num_freq_bins) > 2000)

    # AMOUNT OF EVENTS (0, 1, or 2) AND TIME SHIFT 
    if random.random() < 0.5: # 1 EVENT #0.5
        ev_tmp = np.load(event_file, allow_pickle=True)
        ev_selected, second_event = True, False
    else:
        if random.random() < 0.9:  # NO EVENT #0.9
            ev_tmp = {}
            ev_tmp["processed_waveform_Z"] = np.zeros(8000) # event is 8000 samples long
            ev_selected, second_event = False, False
        else:  # 2 EVENTS
            ev_tmp = np.load(event_file, allow_pickle=True) # event is 8000 samples long
            ev_selected, second_event = True, True
            delta = int(sps * random.uniform(0.01, 0.02500))
            ampl_fac = random.uniform(0.1, 0.9)
    

    # SNR
    if random.random() < 0.33:
        final_snr = random.uniform(min_snr / 2, 100) 
    else:
        if random.random() < 0.5:
            final_snr = random.uniform(min_snr / 2, min_snr)
        else:
            final_snr = random.uniform(min_snr, 1)
    
    snr_ev_1C, std_noise_1C = [], []
    ev_data, noise_data = {}, {}

    comp = "Z"
    noise_snr = noise_tmp["snr"]
    noise_comp = noise_tmp["noise_waveform_" + comp] #6500
    #print(len(noise_comp), len_sample)

    # shift noise start between 0 and 2000 samples
    noise_shift = int(random.uniform(0, 1499))
    noise_comp = noise_comp[noise_shift:noise_shift + len_sample]

    # Process event component
    ev_comp = random.choice([-1, 1]) * ev_tmp["processed_waveform_Z"]

    # Random time shift
    shift_event = int(random.uniform(0,2999))

    # Second event adding process --> to do!!
    if second_event:
        # Adjust delta polarity
        delta = int(np.sign(shift_event - 0.0149 * sps) * delta)

        # Create padded array (original + padding for second event)
        padded_length = len(ev_comp) + 5000
        ev_comp_padded = np.zeros(padded_length)
        
        # Place first event
        ev_comp_padded[2500:2500 + len(ev_comp)] = ev_comp

        # Place second event at delta offset
        ev_comp_padded[2500+delta:2500+delta + len(ev_comp)] += ampl_fac * ev_comp
        # Extract the relevant segment
        ev_comp = ev_comp_padded[2500 + shift_event: 2500 + shift_event + len_sample]
    else:
        ev_comp = ev_comp[shift_event:shift_event + len_sample]

    
    # Compute STFTs
    f, t, ev_stft_comp = scipy.signal.stft(ev_comp, **stft_parameters)
    _, _, noise_stft_comp = scipy.signal.stft(noise_comp, **stft_parameters)

    # (64, 192) by center cropping / padding time bins
    # ev_stft_comp = fix_timebins(ev_stft_comp, 192)
    # noise_stft_comp = fix_timebins(noise_stft_comp, 192)
    
    ev_data["stft_" + comp] = ev_stft_comp
    noise_data["stft_" + comp] = noise_stft_comp

    # Calculate SNR 
    if ev_selected and np.any(np.abs(ev_stft_comp) > 1e-10):  # Check if event has significant energy
        snr_ev_1C.append(np.std(ev_stft_comp)) #[bin_above1Hz:, :]
        std_noise_1C.append(np.std(noise_stft_comp)) #[bin_above1Hz:, :]
        original_snr = np.mean(snr_ev_1C) / (np.mean(std_noise_1C) + 1e-10)  # no division by zero
    else:
        original_snr = 1e-10  # Very small value for no-event cases

    # Scale event
    if ev_selected and original_snr > 1e-8:
        ev_stft_comp_scaled = (ev_data["stft_" + comp] / original_snr) * final_snr
    else:
        ev_stft_comp_scaled = ev_data["stft_" + comp]

    noise_stft_comp_scaled = noise_data["stft_" + comp]

    # Combine event and noise
    noisy_signal = ev_stft_comp_scaled + noise_stft_comp_scaled

    # Compute mask 
    ev_magnitude = np.abs(ev_stft_comp_scaled)
    noise_magnitude = np.abs(noise_stft_comp_scaled)
    
    # Add small epsilon to avoid division by zero
    epsilon = 1e-40
    tmp_mask = ev_magnitude / (ev_magnitude + noise_magnitude + epsilon)
    
    tmp_mask[tmp_mask >= 1] = 1  # safety check
    tmp_mask[tmp_mask <= 0] = 0  # safety check


    # Check for NaN/inf
    if (np.sum(np.isnan(tmp_mask)) > 0 or np.sum(np.isnan(noisy_signal)) > 0 or
            np.sum(np.isinf(tmp_mask)) > 0 or np.sum(np.isinf(noisy_signal)) > 0):
        print('Warning: NaN/inf detected, replacing with zeros')
        noisy_signal = np.zeros_like(noisy_signal)
        tmp_mask = np.zeros_like(tmp_mask)

    # Save data
    dict_save = {}
    dict_save["noisy_signal_" + comp] = np.stack([noisy_signal.real, noisy_signal.imag], axis=-1).astype('float32')
    dict_save["mask" + comp] = tmp_mask.astype('float32')

    if channel_norm:
        noisy_signal_Z = normalize_percentile(dict_save['noisy_signal_Z'], scaler=scaler).astype('float32') #removed clipping
    else:
        noisy_signal_Z = dict_save['noisy_signal_Z']

    mask_Z_temp = dict_save['maskZ'].astype('float32')
    mask_Z = np.zeros([mask_Z_temp.shape[0], mask_Z_temp.shape[1], 1], dtype='float32')
    mask_Z[:, :, 0] = mask_Z_temp

    mask = mask_Z

    # Debug print to check mask statistics
    #print(f"  Event selected: {ev_selected}, Mask range: [{np.min(mask):.3f}, {np.max(mask):.3f}], Mean: {np.mean(mask):.3f}")

    return noisy_signal_Z, mask

@tf.autograph.experimental.do_not_convert
def input_pipeline(data_path_event, data_path_noise, batch_size, len_event, len_noise, clip_val=50, channel_norm=True,scaler="robust"):
    # def input_pipeline(data_path, batch_size):
    """
    Tensorflow's input pipeline for GPU/TPU.
    Loading data +normalize +shuffle +batching in parallel
    Uses map_func for numpy functions
    e.g. load data and normalize
    Input: path to file, batch size
    Output: tensorflor dataset
    """
    dataset_event = tf.data.Dataset.from_tensor_slices(data_path_event)#.repeat()  # takes slices of tensor and create dataset
    dataset_noise = tf.data.Dataset.from_tensor_slices(data_path_noise)#.repeat()  # takes slices of tensor and create dataset

    dataset_event = dataset_event.shuffle(len_event, reshuffle_each_iteration=True)#.repeat()  # randomly shuffles items in dataset CHANGED
    dataset_noise = dataset_noise.shuffle(len_noise, reshuffle_each_iteration=True)#.repeat()  # randomly shuffles items in dataset CHANGED

    dataset = tf.data.Dataset.zip((dataset_event, dataset_noise))
    
    dataset = dataset.map(lambda event, noise: tf.numpy_function(  # transforms items in dataset (with numpy function)
              map_func, [event,noise], [tf.float32,tf.float32]),  # for loading data, normalization
              num_parallel_calls=tf.data.AUTOTUNE) # transformation executed in parallel CHANGED

    dataset = dataset.batch(batch_size, drop_remainder=True)  # CHANGED .repeat()  # combines items into batches #updat
    dataset = dataset.map(lambda x, y: (tf.ensure_shape(x, [None, 64, 192, 2]), tf.ensure_shape(y, [None, 64, 192, 1]))) # [None, 64, 256, components*real+imag] [None, 64, 256, components*(event+noise?)]
    #updated shape to [None, 64, 166, 2] for 1 component (Z) and [None, 64, 166, 1] for mask

    dataset = dataset.prefetch(2) # prepare next batch in parallel # PROBLEMS WITH PREFETCH = AUTOTUNE, GPU USAGE TO HEAVY (?)
    return dataset

def date_name():
  """
  Helper function to get string with date and time
  Input: -
  Output: str
  Requires: datetim
  """
  now = datetime.now()
  return str(now.month).zfill(2) + '_' + str(now.day).zfill(2) + 'T' + str(now.hour).zfill(2) + '_' + str(now.minute).zfill(2)

def conv_block(input, num_filters, batch_norm, kernel_size):
    """
    Conv block with two times: conv layer (+batch norm) + activation
    Input:  data, number of filters and kernel size used in conv2d, bool value if batchnorm required,
    Output: feature maps
    """
    x = Conv2D(num_filters, kernel_size, padding="same")(input)
    if batch_norm:
        x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv2D(num_filters, kernel_size, padding="same")(x)
    if batch_norm:
        x = BatchNormalization()(x)
    x = Activation('relu')(x)

    return x

# Encoder block
# input feature maps of previous block, num_filters: number of output channels
def encoder_block(input, num_filters, batch_norm, pool, kernel_size):
    """
    Encoder block: conv block + pooling
    Input:  data, number of filters and kernel size used in conv2d,
    bool value if batchnorm required, pooling layer (max/avg/conv)
    Output: feature maps before and after pooling
    """
    x = conv_block(input, num_filters, batch_norm,kernel_size)  # calls conv block: conv+(batch+)activation
    if pool=='max':
      p = MaxPool2D((2, 2))(x)  # reduce feature map
    elif pool=='avg':
      p = AveragePooling2D((2, 2))(x)  # reduce feature map
    else:
      p = Conv2D(num_filters, kernel_size, padding="same", strides=2)(x)
      p = Activation('relu')(p)

    return x, p


# Decoder block
# input feature maps of previous block, skip_features: feature maps from encoder branch,
# num_filters: number of output channels
def decoder_block(input, skip_features, num_filters, batch_norm, kernel_size):
    """
    Decoder block: upsampling/transopose conv layer + combine skip connections + conv block
    Input:  data, data from encoder branch (skip_features), number of filters and
    kernel size used in conv2d, bool value if batchnorm required
    Output: feature maps
    """
    x = Conv2DTranspose(num_filters, kernel_size, strides=2, padding="same")(input)  # doubles spatial dimension

    if batch_norm:
      x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Concatenate()([x, skip_features])  # combine with feature maps from encoder
    x = conv_block(x, num_filters, batch_norm, kernel_size)

    return x

# UNET architecture
# input: input_shape = tuple of height, width, number of input channels
def build_unet(input_shape, batch_norm, num_filters_in, pool='max',kernel_size=3):
    """
    Build Unet model
    Input: inpu shape, number of filters and kernel size used in conv2d,
    bool value if batchnorm require, type of pooling layer (max/avg/conv)
    Output: model object
    Requires: conv_block, encoder_block, decoder_block
    """
    inputs = Input(input_shape)  # specify input shape

    # encoder block
    s1, p1 = encoder_block(inputs, num_filters_in, batch_norm, pool, kernel_size)
    s2, p2 = encoder_block(p1, num_filters_in * 2, batch_norm, pool, kernel_size)
    s3, p3 = encoder_block(p2, num_filters_in * 4, batch_norm, pool, kernel_size)
    s4, p4 = encoder_block(p3, num_filters_in * 8, batch_norm, pool, kernel_size)

    # bottleneck
    b1 = conv_block(p4, num_filters_in * 16, batch_norm, kernel_size)

    # decoder block
    d1 = decoder_block(b1, s4, num_filters_in * 8, batch_norm, kernel_size)
    d2 = decoder_block(d1, s3, num_filters_in * 4, batch_norm, kernel_size)
    d3 = decoder_block(d2, s2, num_filters_in * 2, batch_norm, kernel_size)
    d4 = decoder_block(d3, s1, num_filters_in, batch_norm, kernel_size)

    # outputs = Conv2D(6, 1, padding="same", activation="sigmoid")(d4)
    outputs = Conv2D(1, 1, padding="same", activation="sigmoid")(d4) # used to be 3 masks, now 1 mask because 1 component

    model = Model(inputs, outputs, name="unet_small")  # get model obejct
    return model


# Deeper model (example)
def build_unet_deeper(input_shape, batch_norm, num_filters_in, pool='max',kernel_size=3):
    """
    Build Unet model
    Input: inpu shape, number of filters and kernel size used in conv2d,
    bool value if batchnorm require, type of pooling layer (max/avg/conv)
    Output: model object
    Requires: conv_block, encoder_block, decoder_block
    """
    inputs = Input(input_shape)  # specify input shape

    # encoder path
    s1, p1 = encoder_block(inputs, num_filters_in, batch_norm, pool,kernel_size)
    s2, p2 = encoder_block(p1, num_filters_in * 2, batch_norm, pool, kernel_size)
    s3, p3 = encoder_block(p2, num_filters_in * 4, batch_norm, pool, kernel_size)
    s4, p4 = encoder_block(p3, num_filters_in * 8, batch_norm, pool, kernel_size)
    s5, p5 = encoder_block(p4, num_filters_in * 16, batch_norm, pool, kernel_size)

    # bottleneck
    b1 = conv_block(p5, num_filters_in * 32, batch_norm, kernel_size)

    # decoder path
    d1 = decoder_block(b1, s5, num_filters_in * 16, batch_norm, kernel_size)
    d2 = decoder_block(d1, s4, num_filters_in * 8, batch_norm, kernel_size)
    d3 = decoder_block(d2, s3, num_filters_in * 4, batch_norm, kernel_size)
    d4 = decoder_block(d3, s2, num_filters_in * 2, batch_norm, kernel_size)
    d5 = decoder_block(d4, s1, num_filters_in, batch_norm, kernel_size)

    # outputs = Conv2D(6, 1, padding="same", activation="sigmoid")(d5)
    # outputs = Conv2D(3, 1, padding="same", activation="sigmoid")(d5)
    outputs = Conv2D(1, 1, padding="same", activation="softmax")(d5) # 1 mask for event, 1 component

    model = Model(inputs, outputs, name="unet")  # get model obejct
    return model

In [ ]:
train_repeater_factor =  4 #  # repeat training dataset
val_repeater = 4 # repeat validation dataset
batch_size = 32  # mini batch size
num_filters_in = 16 #16  # number of filters in conv
# input_shape = (64,256,6)  # size of sample
input_shape = (64,192,2)  # size of sample

loss = 'binary_crossentropy' # loss function
# opt =  tf.keras.optimizers.Adam(learning_rate=1e-3) # optimizer and lr
opt =  tf.keras.optimizers.Adam(learning_rate=1e-3,clipnorm=1.0) # lr could be 1e-3

# define here path to data
random.seed(123)

# TRAINING data
files_train_event = train_repeater_factor*glob.glob(DATA_DIR + "waveforms/original_dataset_VaMP/train/single_waveforms_snr50/event/Event[0-9][0-9][0-9][0-9][0-9][0-9].npz")
files_train_noise = train_repeater_factor*glob.glob(DATA_DIR + "waveforms/third_train_dataset/single_waveforms_noise_3dayshift/noise/Noise[0-9][0-9][0-9][0-9][0-9][0-9].npz")
print(len(files_train_event), len(files_train_noise))
if len(files_train_event) < len(files_train_noise):
    # Calculate how many times to repeat the noise files
    repeat_times = int(np.ceil(len(files_train_noise) / len(files_train_event)))
    # Repeat the noise files list
    files_train_event = files_train_event * repeat_times
    # Trim to exactly match the event files length
    files_train_event = files_train_event[:len(files_train_noise)]

# Validation data
files_val_event = val_repeater*glob.glob(DATA_DIR + "waveforms/second_val_test_dataset/single_waveforms_snr50/val/event/Event[0-9][0-9][0-9][0-9][0-9][0-9].npz")
files_val_noise = val_repeater*glob.glob(DATA_DIR + "waveforms/third_val_test_dataset/single_waveforms_noise_3dayshift/val/noise/Noise[0-9][0-9][0-9][0-9][0-9][0-9].npz")
if len(files_val_event) < len(files_val_noise):
    # Calculate how many times to repeat the noise files
    repeat_times = int(np.ceil(len(files_val_noise) / len(files_val_event)))
    # Repeat the noise files list
    files_val_event = files_val_event * repeat_times
    # Trim to exactly match the event files length
    files_val_event = files_val_event[:len(files_val_noise)]

# Generate dataset for training and testing
train_dataset = input_pipeline(files_train_event, files_train_noise,batch_size,len_event=len(files_train_event),len_noise=len(files_train_noise))
val_dataset = input_pipeline(files_val_event, files_val_noise, batch_size, len_event=len(files_val_event), len_noise=len(files_val_noise))

with tf.device('/gpu:0'):  # use GPU
  # build model
  #model = build_unet_deeper(input_shape,batch_norm=False,num_filters_in=num_filters_in, pool='max',kernel_size=3)
  model = build_unet(input_shape,batch_norm=False,num_filters_in=num_filters_in, pool='max',kernel_size=3)

# compile model
model.compile(optimizer=opt, loss=loss, metrics=[loss])

print('Selected:')
print('Train samples event: ' + str(len(files_train_event)))
print('Train samples noise: ' + str(len(files_train_noise)))

print('Validation samples event: ' + str(len(files_val_event)))
print('Validation samples noise: ' + str(len(files_val_noise)))

In [ ]:
# for j in range(1):
#     for images, labels in train_dataset.take(1):  # only take first element of dataset
#         x = images.numpy()
#         y = labels.numpy()

In [ ]:
#     # # %%
#     # for images, labels, weights in withweights.take(1):  # only take first element of dataset
#     #     x = images.numpy()
#     #     y = labels.numpy()
#     #     w = weights.numpy()
#
#     # # # # %%
#     plt.subplot(121)
#     plt.pcolormesh(x[0,:,:,0],vmin=-5,vmax=5,cmap="bwr")
#     plt.colorbar()
#     plt.subplot(122)
#     plt.pcolormesh(y[0,:,:,0],vmin=0,vmax=1,cmap="cubehelix_r")
#     plt.colorbar()
#     plt.show()
# #     # plt.pcolormesh(w[0,:,:,0])
# #     # plt.colorbar()
# #     # plt.show()
# #     # # # # train_dataset.map(add_sample_weights)

# plt.subplot(121)
# plt.pcolormesh(x[0, :, :, 5], vmin=-5, vmax=5, cmap="bwr")
# plt.colorbar()
# plt.subplot(122)
# plt.pcolormesh(y[0, :, :, 2], vmin=0, vmax=1, cmap="cubehelix_r")
# plt.colorbar()
# plt.show()

In [ ]:
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',       # Metric to monitor, often 'val_loss' or 'val_accuracy'
    factor=0.1,               # Factor by which the learning rate will be reduced (new_lr = lr * factor)
    patience=3,              # Number of epochs with no improvement after which learning rate will be reduced (was 5)
    min_delta=0.001,            
    min_lr=1e-6,              # Lower bound on the learning rate to prevent it from going too low
    cooldown=0,               # Number of epochs to wait after a reduction before resuming normal operation
    verbose=1
)

In [ ]:
# visualize samples from training dataset, changes 3 components to 1 component

# tf.keras.utils.plot_model(model,show_shapes=True)

model.summary()
# # # %%
# take a couple of examples (here images=stft coefficients, labels=masks)
for images, labels in train_dataset.take(1):  # only take first element of dataset
    x = images.numpy()
    y = labels.numpy()


def label_off(ax):
    # helper function to remove tick labels in plot
    ax.axes.get_xaxis().set_ticks([])
    ax.axes.get_yaxis().set_ticks([])

# figure specs
num_events = np.shape(x)[0]
fig = plt.figure(figsize=[24, 40])
spec = gridspec.GridSpec(ncols=6, nrows=num_events, figure=fig)  # ,height_ratios=[1,1,1,1])
spec.update(wspace=0.075, hspace=0.1)

for i in range(num_events):
    f_ax1 = fig.add_subplot(spec[i, 0])
    f_ax1.pcolormesh(x[i, :, :, 0], vmin=-3, vmax=3, cmap='bwr')
    label_off(f_ax1)
    f_ax1.set_ylabel('0-50 Hz')

    f_ax2 = fig.add_subplot(spec[i, 1])
    f_ax2.pcolormesh(x[i, :, :, 1], vmin=-3, vmax=3, cmap='bwr')
    label_off(f_ax2)

    f_ax3 = fig.add_subplot(spec[i, 2])
    f_ax3.pcolormesh(y[i, :, :, 0], vmin=0, vmax=1, cmap='cubehelix_r')
    label_off(f_ax3)

    if i == num_events - 1:
        for ax in fig.axes:
            ax.set_xlabel('0.5 sec s')

#plt.suptitle('Z-input (real) Z-input (imag)  Z-target (event)', y=0.05, fontsize=20)
plt.show()

In [ ]:
model_name = 'zcomp_unet_192x64_seventhrun_17_09_2025_trainrep4_valrep4' # name model

print(model_name)
model_path = DATA_DIR + "models/" # folder were model weights are strored
dir_history = DATA_DIR + "history/" # folder to store training history

print("Train samples: " + str(len(files_train_event)))
print("Val samples: " + str(len(files_val_event)))

print("Training batch iteration/epoch: " + str(evrepeater_factor*int(len(files_train_event) / batch_size)))
print("Validation batch iteration/epoch: " + str(evrepeater_factor*int(len(files_val_event) / batch_size)))

# define checkpoints, saves best model weights (only if val_loss improves)
model_checkpoint_callback = ModelCheckpoint(model_path+model_name+'.keras',  # '-{epoch:04d}.h5'
                             verbose=1, monitor='val_loss',save_best_only=True, mode='auto')

# define eartly stopping and criteria (e.g. validaion loss not improving for 5 epochs by more than 0.001)
callback = EarlyStopping(monitor='val_loss', patience=5,min_delta=1e-5,restore_best_weights=True)

start_time = time.time()
with tf.device('/gpu:0'):

    history = model.fit(train_dataset, epochs=500, verbose=1,
                        batch_size=batch_size,validation_data=val_dataset,
                        callbacks=[model_checkpoint_callback, callback,reduce_lr])

print("--- %s seconds ---" % (time.time() - start_time))

# save model
model_name += date_name()  # get name as current time
dir_history +=model_name  # get dir
np.save(dir_history+'.npy',history.history)  # and save

In [ ]:

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Training/Validation Loss of Model; ' + model_name)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train', 'Validation'])
plt.tight_layout()
plt.grid()
plt.show()

In [ ]:

# Path to your history file
history_path = DATA_DIR + "history/zcomp_unet_192x64_fourthrun_14_08_2025_repeater2.npy"
model_name = "zcomp_unet_192x64_sixthrun_16_08_2025_evr6_valr6"
# Load the history file
history_dict = np.load(history_path, allow_pickle=True).item()



In [ ]:
print(history_dict)
print(np.min(history_dict['val_loss']))

In [ ]:
plt.plot(history_dict['loss'])
plt.plot(history_dict['val_loss'])
plt.plot(history_dict['lr'])
plt.title('Training/Validation Loss of Model; ' + model_name)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(['Train loss', 'Validation loss', 'Learning rate'])
plt.tight_layout()
plt.grid()
plt.show()

## visualisation of denoising performance below, see last cell for figures


In [ ]:
# def normalize_percentile2(
#     data, 
#     event,
#     quantile_range=(25, 75),
#     unit_variance=False,
#     limit=50,
#     clipping='none',
#     scaler="standard"
# ):

#     # Separate real and imaginary parts
#     data_real = data[:, :, 0]
#     data_imag = data[:, :, 1]
#     event_real = event[:, :, 0]
#     event_imag = event[:, :, 1]

#     freq_bins, time_bins = data_real.shape

#     # Choose scaler type
#     if scaler == "robust":
#         scaler_real = RobustScaler(unit_variance=unit_variance)
#         scaler_imag = RobustScaler(unit_variance=unit_variance)
#     else:
#         scaler_real = StandardScaler()
#         scaler_imag = StandardScaler()

#     #scaler_real.fit(data_real.flatten()[:, None])

#     # Apply the same scaler to both noisy and event data
#     data_real_norm = scaler_real.fit_transform(data_real.flatten()[:, None]).reshape(freq_bins, time_bins)

#     event_real_norm = scaler_real.fit_transform(event_real.flatten()[:, None]).reshape(freq_bins, time_bins)

#     #scaler_imag.fit(data_imag.flatten()[:, None])
#     # Apply the same scaler to both noisy and event data
#     data_imag_norm = scaler_imag.fit_transform(data_imag.flatten()[:, None]).reshape(freq_bins, time_bins)
    
#     event_imag_norm = scaler_imag.fit_transform(event_imag.flatten()[:, None]).reshape(freq_bins, time_bins)

#     # Optional clipping
#     if clipping == 'constant':
#         for arr in (data_real_norm, data_imag_norm):
#             arr[arr >  limit] =  limit
#             arr[arr < -limit] = -limit

#     # Combine real and imag back into output arrays
#     data_return = np.zeros_like(data)
#     data_return[:, :, 0] = data_real_norm
#     data_return[:, :, 1] = data_imag_norm

#     event_return = np.zeros_like(event)
#     event_return[:, :, 0] = event_real_norm
#     event_return[:, :, 1] = event_imag_norm

#     return data_return, event_return


In [ ]:
# def normalize_percentile2(
#     data, 
#     event,
#     quantile_range=(25, 75),
#     unit_variance=False,
#     limit=50,
#     clipping='none',
#     scaler="standard"
# ):
#     # Separate real and imaginary parts
#     data_real = data[:, :, 0]
#     data_imag = data[:, :, 1]
#     event_real = event[:, :, 0]
#     event_imag = event[:, :, 1]

#     freq_bins, time_bins = data_real.shape

#     # Choose scaler type
#     if scaler == "robust":
#         scaler_real = RobustScaler(unit_variance=unit_variance)
#         scaler_imag = RobustScaler(unit_variance=unit_variance)
#     else:
#         scaler_real = StandardScaler()
#         scaler_imag = StandardScaler()

#     # Fit scaler only on noisy data, then transform both
#     scaler_real.fit(data_real.flatten()[:, None])
#     data_real_norm = scaler_real.transform(data_real.flatten()[:, None]).reshape(freq_bins, time_bins)
#     event_real_norm = scaler_real.transform(event_real.flatten()[:, None]).reshape(freq_bins, time_bins)

#     scaler_imag.fit(data_imag.flatten()[:, None])
#     data_imag_norm = scaler_imag.transform(data_imag.flatten()[:, None]).reshape(freq_bins, time_bins)
#     event_imag_norm = scaler_imag.transform(event_imag.flatten()[:, None]).reshape(freq_bins, time_bins)

#     # Optional clipping
#     if clipping == 'constant':
#         for arr in (data_real_norm, data_imag_norm):
#             arr[arr >  limit] =  limit
#             arr[arr < -limit] = -limit

#     # Combine real and imag back into output arrays
#     data_return = np.zeros_like(data)
#     data_return[:, :, 0] = data_real_norm
#     data_return[:, :, 1] = data_imag_norm

#     event_return = np.zeros_like(event)
#     event_return[:, :, 0] = event_real_norm
#     event_return[:, :, 1] = event_imag_norm

#     return data_return, event_return

In [ ]:
def map_func_plotting(event_file, noise_file, clip_val=1000, channel_norm=True, sps=100000, scaler="robust"):
    """
    Mapping (numpy) function in input_pipeline: load data and normalise
    """
    len_sample = 4967
    noise_tmp = np.load(noise_file, allow_pickle=True) #6500 samples long
    min_snr = 0.2 
    delta = 0
    ampl_fac = None # TEMP
    
    stft_parameters = {"nperseg": 64, "nfft": 126, "fs": 100000, "noverlap": 38, "boundary": "zeros", "padded": False}
    #num_freq_bins = int(stft_parameters["nfft"] / 2 + 1)
    #bin_above1Hz = np.argmax(np.linspace(0, 50000, num_freq_bins) > 2000)

    # AMOUNT OF EVENTS (0, 1, or 2) AND TIME SHIFT ---> double event cases todo!
    if random.random() < 0.5: # 1 EVENT #0.5
        ev_tmp = np.load(event_file, allow_pickle=True)
        ev_selected, second_event = True, False
    else:
        if random.random() < 0.9:  # NO EVENT #0.9
            ev_tmp = {}
            ev_tmp["processed_waveform_Z"] = np.zeros(8000) # event is 8000 samples long
            ev_selected, second_event = False, False
        else:  # 2 EVENTS
            ev_tmp = np.load(event_file, allow_pickle=True) # event is 8000 samples long
            ev_selected, second_event = True, True
            delta = int(sps * random.uniform(0.01, 0.02469))
            ampl_fac = random.uniform(0.1, 0.9)
    

    # SNR
    if random.random() < 0.33:
        final_snr = random.uniform(min_snr / 2, 100)
    else:
        if random.random() < 0.5:
            final_snr = random.uniform(min_snr / 2, min_snr)
        else:
            final_snr = random.uniform(min_snr, 1)
    
    snr_ev_1C, std_noise_1C = [], []
    ev_data, noise_data = {}, {}

    comp = "Z"
    noise_snr = noise_tmp["snr"]
    noise_comp = noise_tmp["noise_waveform_" + comp] 
    #print(len(noise_comp), len_sample)

    # shift noise start between 0 and 1500 samples
    noise_shift = int(random.uniform(0, 1500))
    noise_comp = noise_comp[noise_shift:noise_shift + len_sample]

    # Process event component
    ev_comp = random.choice([-1, 1]) * ev_tmp["processed_waveform_Z"]

    # Random time shift
    shift_event = int(random.uniform(0,2999))
     
    # Second event adding process 
    if second_event:
        # Adjust delta polarity
        delta = int(np.sign(shift_event - 0.0150 * sps) * delta)
        
        # Create padded array (original + padding for second event)
        padded_length = len(ev_comp) + 4967
        ev_comp_padded = np.zeros(padded_length)
        
        # Place first event
        ev_comp_padded[2500:2500 + len(ev_comp)] = ev_comp

        # Place second event at delta offset
        ev_comp_padded[2500 + delta:2500 + delta + len(ev_comp)] += ampl_fac * ev_comp
        # Extract the relevant segment
        ev_comp = ev_comp_padded[2500 + shift_event: 2500 + shift_event + len_sample]
    else:
        ev_comp = ev_comp[shift_event:shift_event + len_sample]

    print(np.max(ev_comp),np.max(noise_comp))
    # Compute STFTs
    f, t, ev_stft_comp = scipy.signal.stft(ev_comp, **stft_parameters)
    _, _, noise_stft_comp = scipy.signal.stft(noise_comp, **stft_parameters)

    # (64, 192) by center cropping / padding time bins
    #ev_stft_comp = fix_timebins(ev_stft_comp, 192)
    #noise_stft_comp = fix_timebins(noise_stft_comp, 192)
    
    ev_data["stft_" + comp] = ev_stft_comp
    noise_data["stft_" + comp] = noise_stft_comp

    # Calculate SNR 
    if ev_selected and np.any(np.abs(ev_stft_comp) > 1e-10):  # Check if event has significant energy
        snr_ev_1C.append(np.std(ev_stft_comp)) #[bin_above1Hz:, :]
        std_noise_1C.append(np.std(noise_stft_comp)) #[bin_above1Hz:, :]
        original_snr = np.mean(snr_ev_1C) / (np.mean(std_noise_1C) + 1e-10)  # no division by zero
    else:
        original_snr = 1e-10  # Very small value for no-event cases

    # Scale event
    if ev_selected and original_snr > 1e-8:
        ev_stft_comp_scaled = (ev_data["stft_" + comp] / original_snr) * final_snr
    else:
        ev_stft_comp_scaled = ev_data["stft_" + comp]

    noise_stft_comp_scaled = noise_data["stft_" + comp]

    # Combine event and noise
    noisy_signal = ev_stft_comp_scaled + noise_stft_comp_scaled

    # Compute mask 
    ev_magnitude = np.abs(ev_stft_comp_scaled)
    noise_magnitude = np.abs(noise_stft_comp_scaled)
    
    # Add small epsilon to avoid division by zero
    epsilon = 1e-40
    tmp_mask = ev_magnitude / (ev_magnitude + noise_magnitude + epsilon)
    
    tmp_mask[tmp_mask >= 1] = 1  # safety check
    tmp_mask[tmp_mask <= 0] = 0  # safety check

    # Check for NaN/inf
    if (np.sum(np.isnan(tmp_mask)) > 0 or np.sum(np.isnan(noisy_signal)) > 0 or
            np.sum(np.isinf(tmp_mask)) > 0 or np.sum(np.isinf(noisy_signal)) > 0):
        print('Warning: NaN/inf detected, replacing with zeros')
        noisy_signal = np.zeros_like(noisy_signal)
        tmp_mask = np.zeros_like(tmp_mask)

    # # Save data
    # dict_save = {}
    # dict_save["noisy_signal_" + comp] = np.stack([noisy_signal.real, noisy_signal.imag], axis=-1).astype('float32')
    # dict_save["mask" + comp] = tmp_mask.astype('float32')
    
    # event_waveform = {}
    # event_waveform["event_waveform"] = np.stack([ev_stft_comp_scaled.real, ev_stft_comp_scaled.imag], axis=-1).astype('float32')
    # # print(event_waveform.shape())
    # # print(noisy_signal.shape())

    # # normalize with same scaler for both event and noisy signal
    # if channel_norm:
    #     event_waveform, noisy_signal_Z = normalize_percentile2(
    #     event_waveform["event_waveform"], dict_save["noisy_signal_Z"], scaler=scaler)
    # else:
    #     event_waveform = event_waveform
    #     noisy_signal_Z = noisy_signal

    # mask_Z_temp = dict_save['maskZ'].astype('float32')
    # mask_Z = np.zeros([mask_Z_temp.shape[0], mask_Z_temp.shape[1], 1], dtype='float32')
    # mask_Z[:, :, 0] = mask_Z_temp

    # mask = mask_Z

    # Debug print to check mask statistics
    #print(f"  Event selected: {ev_selected}, Mask range: [{np.min(mask):.3f}, {np.max(mask):.3f}], Mean: {np.mean(mask):.3f}")

    # Save data
    dict_save = {}
    dict_save["noisy_signal_" + comp] = np.stack([noisy_signal.real, noisy_signal.imag], axis=-1).astype('float32')
    dict_save["mask" + comp] = tmp_mask.astype('float32')
    
    # First normalize the noisy data exactly as done during training
    if channel_norm:
        noisy_signal_Z = normalize_percentile(dict_save['noisy_signal_Z'], scaler=scaler).astype('float32')
    else:
        noisy_signal_Z = dict_save['noisy_signal_Z']
    
    # Then normalize event separately but using the same parameters
    event_waveform = {}
    event_waveform["event_waveform"] = np.stack([ev_stft_comp_scaled.real, ev_stft_comp_scaled.imag], axis=-1).astype('float32')
    
    # Extract scalers from noisy data for real and imaginary parts
    data_real = dict_save["noisy_signal_" + comp][:, :, 0]
    data_imag = dict_save["noisy_signal_" + comp][:, :, 1]
    
    # Create and fit scalers on noisy data only
    if scaler == "robust":
        scaler_real = RobustScaler()
        scaler_imag = RobustScaler()
    else:
        scaler_real = StandardScaler()
        scaler_imag = StandardScaler()

     # Fit scalers on noisy data
    scaler_real.fit(data_real.flatten()[:, None])
    scaler_imag.fit(data_imag.flatten()[:, None])
    
    # Apply to event data
    event_real = event_waveform["event_waveform"][:, :, 0]
    event_imag = event_waveform["event_waveform"][:, :, 1]
    freq_bins, time_bins = event_real.shape
    
    event_real_norm = scaler_real.transform(event_real.flatten()[:, None]).reshape(freq_bins, time_bins)
    event_imag_norm = scaler_imag.transform(event_imag.flatten()[:, None]).reshape(freq_bins, time_bins)
    
    event_normalized = np.zeros_like(event_waveform["event_waveform"])
    event_normalized[:, :, 0] = event_real_norm
    event_normalized[:, :, 1] = event_imag_norm
    
    # Mask processing
    mask_Z_temp = dict_save['maskZ'].astype('float32')
    mask_Z = np.zeros([mask_Z_temp.shape[0], mask_Z_temp.shape[1], 1], dtype='float32')
    mask_Z[:, :, 0] = mask_Z_temp
    
    mask = mask_Z
    

    return noisy_signal_Z, mask, ev_selected, event_normalized, final_snr

In [ ]:
model_path = DATA_DIR + "model_history/models/zcomp_unet_192x64_seventhrun_17_09_2025_trainrep4_valrep4.keras"

model = tf.keras.models.load_model(model_path)  # load model

def process_data2(model, noisy_signal_stft, mask_act, ev_selected, ev_stft_comp_scaled, final_snr, output_path=None): 
    
    
    model_input = np.expand_dims(noisy_signal_stft, axis=0)  # add batch dimension
    #ev_stft_comp_scaled = np.expand_dims(ev_stft_comp_scaled, axis=0)
    
    # predict mask
    with tf.device('/gpu:0'):
        predicted_mask = model.predict(model_input)
    
    # extract 2D mask
    predicted_mask_2d = predicted_mask[0, :, :, 0]
    
    plt.figure(figsize=(16, 8))
    
    # real part input spectrogram
    plt.subplot(231)
    plt.pcolormesh(model_input[0, :, :, 0], cmap='bwr', vmin=-3, vmax=3)
    plt.colorbar()
    plt.title('Input Spectrogram (Real)')
    plt.xlabel('Time bins')
    plt.ylabel('Freq bins')
    
    # imaginary part input spectrogram
    plt.subplot(232)
    plt.pcolormesh(model_input[0, :, :, 1], cmap='bwr', vmin=-3, vmax=3)
    plt.colorbar()
    plt.title('Input Spectrogram (Imaginary)')
    plt.xlabel('Time bins')
    plt.ylabel('Freq bins')
    
    # actual mask
    plt.subplot(234)
    plt.pcolormesh(mask_act[:, :, 0], cmap='cubehelix_r', vmin=0, vmax=1)
    plt.colorbar()
    plt.title('Actual Mask')
    plt.xlabel('Time bins')
    plt.ylabel('Freq bins')

    # predicted mask
    plt.subplot(235)
    plt.pcolormesh(predicted_mask_2d, cmap='cubehelix_r', vmin=0, vmax=1)
    plt.colorbar()
    plt.title('Predicted Mask')
    plt.xlabel('Time bins')
    plt.ylabel('Freq bins')

    # complex stft from real and imaginary parts noisy signal
    noisy_stft_complex = noisy_signal_stft[:, :, 0] + 1j * noisy_signal_stft[:, :, 1]
    ev_stft_complex = ev_stft_comp_scaled[:, :, 0] + 1j * ev_stft_comp_scaled[:, :, 1]
    
    # element-wise multiplication 
    denoised_stft = noisy_stft_complex * predicted_mask_2d
    
    # ISTFT parameters
    # ISTFT parameters - use the same as STFT
    istft_parameters = {"nperseg": 64, "nfft": 126, "fs": 100000, "noverlap": 38, "boundary": "zeros"}
    
    # denoised waveform time domain
    t_rec, waveform_denoised = scipy.signal.istft(denoised_stft, **istft_parameters)
    
    #original waveform
    t_orig, original_waveform = scipy.signal.istft(noisy_stft_complex, **istft_parameters)

    t_ev, event_waveform = scipy.signal.istft(ev_stft_complex, **istft_parameters)
    print(np.max(waveform_denoised), np.max(original_waveform), np.max(event_waveform))
    print(len(waveform_denoised), len(original_waveform), len(event_waveform))

    # original vs denoised waveform

    if ev_selected:
        plt.subplot(233)
        plt.plot(original_waveform, color='gray', alpha=0.4, label='Noisy waveform', linestyle='--')
        plt.plot(event_waveform, color='b', label='Event waveform', alpha=0.4, linestyle='-')
        plt.plot(waveform_denoised, color='r', label='Denoised waveform', alpha=0.4, linestyle='-')
        plt.legend()
        plt.title(f'Waveform Comparison (With Event), SNR: {final_snr:.2f}')
        plt.xlabel('Samples')
        plt.ylabel('Counts')
    else:
        plt.subplot(233)
        plt.plot(original_waveform, color='gray', alpha=0.4, label='Noisy waveform', linestyle='--')
        plt.plot(waveform_denoised, color='r', label='Denoised waveform', alpha=0.7, linestyle='-')
        plt.legend()
        plt.title('Waveform Comparison (No Event)')
        plt.xlabel('Samples')
        plt.ylabel('Counts')

    # mask difference
    plt.subplot(236)
    mask_diff = np.abs(mask_act[:, :, 0] - predicted_mask_2d)
    plt.pcolormesh(mask_diff, cmap='cubehelix_r', vmin=0, vmax=1)
    plt.colorbar()
    plt.title('Mask Difference (|actual-predicted|)')
    plt.xlabel('Time bins')
    plt.ylabel('Freq bins')
    plt.tight_layout()
    
    # if output_path:
    #     # Save results - Correct indexing for 3D array
    #     np.savez(output_path, 
    #              original_waveform=original_waveform,
    #              denoised_waveform=waveform_denoised,
    #              actual_mask=mask_act[:, :, 0],
    #              predicted_mask=predicted_mask_2d)
    #     plt.savefig(output_path.replace('.npz', '.png'))
    
    plt.show()
    
    return original_waveform, waveform_denoised, predicted_mask_2d

In [ ]:
import random
import glob
#import warnings
#warnings.filterwarnings('ignore', message="NOLA condition failed")

#testing files
event_files = val_repeater*glob.glob(DATA_DIR + "waveforms/second_val_test_dataset/single_waveforms_snr50/test/event/Event[0-9][0-9][0-9][0-9][0-9][0-9].npz")
noise_files = val_repeater*glob.glob(DATA_DIR + "waveforms/third_val_test_dataset/single_waveforms_noise_3dayshift/test/noise/Noise[0-9][0-9][0-9][0-9][0-9][0-9].npz")

# Select a random event and noise file
event_file = random.choice(event_files)
noise_file = random.choice(noise_files)

noisy_signal, mask_act, ev_selected, evstft, final_snr = map_func_plotting(event_file, noise_file)


#output_path = DATA_DIR + "results/denoised_event1.npz"

# process the data
original, denoised, mask_pred = process_data2(model, noisy_signal, mask_act, ev_selected, evstft, final_snr)